# Deep Agents на GigaChat — практический блокнот

[Deep Agents](https://github.com/langchain-ai/deepagents) — агентный харнесс от LangChain поверх LangGraph: агент получает файловую систему, shell, планировщик, субагентов и навыки «из коробки». Этот блокнот показывает его в связке с **GigaChat** на сквозном примере агента-аналитика, который читает данные, пишет и запускает код и оформляет отчёт.

Что демонстрируем:
- разминку с виртуальной файловой системой (`StateBackend`) — без единого своего инструмента;
- реальную работу с файлами и shell (`LocalShellBackend`) на живых данных;
- навыки (skills) с прогрессивным раскрытием;
- наблюдаемость через Arize Phoenix.

**Порядок:** окружение → конфиг из `.env` → наблюдаемость → разминка → аналитик с shell → навыки.

> Профиль харнесса [`deepagents-gigachat`](https://github.com/ai-forever/deepagents-gigachat) подключается автоматически (через entry point `deepagents.harness_profiles`) — достаточно, чтобы пакет был установлен. Он подстраивает под особенности GigaChat системный промпт и описания инструментов и добавляет своё middleware; здесь мы просто им пользуемся.


## 1. Окружение

Зависимости ставятся через **uv** (как в репозиториях команды), в изолированный `.venv` с **Python 3.12+** — базовый Python не трогаем. Из папки блокнота:

```bash
uv sync             # создаст .venv (Python 3.12+) и поставит всё из pyproject.toml
uv run jupyter lab  # запустит Jupyter из этого окружения
```

В VS Code выберите интерпретатор/ядро из `.venv`. Отдельная установка внутри блокнота не нужна — всё уже в окружении. Профиль `deepagents-gigachat` (нужен Python ≥ 3.12 и `deepagents` ≥ 0.6.7) уже в зависимостях.


## 2. Конфигурация GigaChat из `.env`

Скопируйте `.env.example` → `.env` и заполните (авторизационный ключ — на [developers.sber.ru](https://developers.sber.ru/docs/ru/gigachat/individuals-quickstart) → ваш проект). Авторизацию, scope и base_url и другие параметры `langchain-gigachat` подхватит из окружения сам.


In [1]:
import os

from dotenv import load_dotenv
from langchain_gigachat import GigaChat

load_dotenv()

def env_bool(name, default=False):
    return os.getenv(name, str(default)).strip().lower() in ("1", "true", "yes", "on")

llm = GigaChat()

# Быстрая проверка, что авторизация и модель живые:
print(llm.invoke("Ответь одним словом: работает?").content)

Да


## 3. Наблюдаемость: Phoenix

Чем толще обвязка, тем важнее видеть, что внутри. У deep-агента за один запуск — десятки вызовов модели и инструментов; по одному stdout понять, где он свернул не туда, почти нереально. [**Arize Phoenix**](https://github.com/Arize-ai/phoenix) — открытый локальный инструмент трейсинга для LLM-приложений: в его UI видно всё дерево шагов агента — каждый вызов модели и инструмента, содержимое контекста, расход токенов, тайминги, отдельные ветки субагентов.

Поднимаем Phoenix **до** первого прогона, чтобы все шаги ниже сразу попали в трейс, и держим его **отдельным процессом**, а не внутри ядра: так рестарт ядра его не роняет, порт `6006` не залипает, а трейсы копятся в персистентной БД (`~/.phoenix`). В отдельном терминале из папки блокнота:

```bash
uv run phoenix serve
```

UI откроется на `http://localhost:6006`. Ячейка ниже сервер **не** поднимает — она лишь инструментирует LangChain: `register(...)` регистрирует OpenTelemetry-трейсер на коллектор Phoenix (`endpoint`), а `auto_instrument=True` сам подключает [OpenInference](https://github.com/Arize-ai/openinference)-обёртку для LangChain. После этого каждый прогон агента ниже автоматически шлёт трейсы в Phoenix, и ячейку можно перезапускать сколько угодно.


In [2]:
from phoenix.otel import register

register(
    project_name="deepagents-gigachat",
    endpoint="http://localhost:6006/v1/traces",
    auto_instrument=True,
)
print("Трейсы идут в Phoenix на http://localhost:6006")

<project>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: deepagents-gigachat
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Трейсы идут в Phoenix на http://localhost:6006


## 4. Минимальный агент: файловая система в памяти

Начнём с агента, которому не даём ни одного своего инструмента. Попросим составить план и сохранить его в файл. Файловые инструменты (`ls`, `read_file`, `write_file`, …) у deep-агента есть по умолчанию, и без указания бэкенда они пишут в `StateBackend` — состояние графа LangGraph, а не на диск. Файл появится в `result["files"]` под ключом `/plan.md` (виртуальная файловая система deepagents отсчитывает пути от своего корня `/`), а на машине не останется никаких следов.


In [2]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=llm, system_prompt="Ты полезный ассистент.")

result = agent.invoke({"messages": [{"role": "user", "content":
    "Составь план изучения LangGraph на неделю и сохрани его в файл plan.md"}]})

for m in result["messages"]:
    m.pretty_print()

files = result.get("files", {})
print("\nФайлы в состоянии:", list(files.keys()))

[deepagents-gigachat] Harness profile loaded (providers=gigachat,giga; variant=native_fs; tool_contract=off)


================================ Human Message =================================

Составь план изучения LangGraph на неделю и сохрани его в файл plan.md
================================== Ai Message ==================================
Tool Calls:
  write_file (9dd02ebb-eca9-46ff-806e-f25e2c37201f)
 Call ID: 9dd02ebb-eca9-46ff-806e-f25e2c37201f
  Args:
    content: # План изучения LangChain на неделю

## Понедельник
- Введение в LangChain и экосистему
- Установка и настройка окружения
- Создание первого простого LLM-цепа

## Вторник
- Работа с PromptTemplate и форматированием запросов
- Использование различных LLM-провайдеров
- Обработка ответов и форматирование вывода

## Среда
- Работа с памятью в цепях (Memory)
- Создание контекстно-зависимых диалогов
- Управление состоянием цепи

## Четверг
- Интеграция с внешними инструментами и API
- Работа с документами и загрузчиками
- Векторные хранилища и поиск по документам

## Пятница
- Цепи с агентами (Agents)
- Создание собственных инструме

## 5. Аналитик + shell на реальных данных

Теперь у агента появляются файлы и shell. Готовим песочницу с `sales.csv` и подключаем `LocalShellBackend` — файловые инструменты плюс `execute` (shell). Агент сам изучит данные, напишет `analyze.py`, запустит его и оформит `report.md`; числа берутся из вывода реально выполненного кода, а не «из головы». Профиль GigaChat подталкивает агента к относительным путям — это важно при `virtual_mode=True`.

> `LocalShellBackend` выполняет команды прямо на вашей машине. Используйте его только в доверенной локальной среде; для продакшена есть sandbox-бэкенды с изоляцией.


In [4]:
import csv
import random
import shutil
from pathlib import Path

workspace = Path("workspace")
shutil.rmtree(workspace, ignore_errors=True)   # чистим песочницу перед прогоном
workspace.mkdir(exist_ok=True)

random.seed(42)
with open(workspace / "sales.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["date", "region", "product", "qty", "price"])
    for month in range(1, 7):
        for _ in range(50):
            w.writerow([
                f"2026-{month:02d}-{random.randint(1, 28):02d}",
                random.choice(["Москва", "СПб", "Казань", "Новосибирск"]),
                random.choice(["A", "B", "C"]),
                random.randint(1, 20),
                random.choice([990, 1490, 2490]),
            ])

print("sales.csv готов:", (workspace / "sales.csv").stat().st_size, "байт")

sales.csv готов: 10808 байт


In [5]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import MemorySaver

backend = LocalShellBackend(root_dir=str(workspace.resolve()), virtual_mode=True)

agent = create_deep_agent(
    model=llm,
    backend=backend,
    checkpointer=MemorySaver(),
    system_prompt=(
        "Ты — аналитик данных с доступом к файлам и shell. "
        "Работай итеративно: изучи данные, напиши код, запусти его, проверь результат. "
        "Используй относительные пути. Не выдумывай цифры — бери их только из вывода своих скриптов."
    ),
)

config = {"configurable": {"thread_id": "sales-analysis"}}
result = agent.invoke({"messages": [{"role": "user", "content":
    "В рабочей папке лежит sales.csv. Разберись в структуре данных, напиши Python-скрипт "
    "analyze.py, который считает выручку по регионам и по месяцам, запусти его и оформи "
    "отчёт report.md с таблицами и выводами."}]}, config=config)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

В рабочей папке лежит sales.csv. Разберись в структуре данных, напиши Python-скрипт analyze.py, который считает выручку по регионам и по месяцам, запусти его и оформи отчёт report.md с таблицами и выводами.
================================== Ai Message ==================================
Tool Calls:
  read_file (f7b9aaf5-8cb7-4675-8332-bdf3e41898ee)
 Call ID: f7b9aaf5-8cb7-4675-8332-bdf3e41898ee
  Args:
    file_path: sales.csv
================================= Tool Message =================================
Name: read_file

     1	date,region,product,qty,price
     2	2026-01-21,Москва,A,9,990
     3	2026-01-08,СПб,C,4,2490
     4	2026-01-24,Москва,C,14,990
     5	2026-01-01,Москва,A,8,2490
     6	2026-01-20,Москва,C,7,2490
     7	2026-01-21,Новосибирск,A,15,2490
     8	2026-01-09,Москва,A,14,1490
     9	2026-01-09,СПб,A,11,990
    10	2026-01-03,Новосибирск,A,12,1490
    11	2026-01-20,Казань,A,15,2490
    1

In [6]:
# Что осталось в песочнице после прогона:
for p in sorted(workspace.glob("*")):
    print(p.name, "-", p.stat().st_size, "байт")

report = workspace / "report.md"
print("\n===== report.md =====\n")
print(report.read_text(encoding="utf-8") if report.exists() else "(report.md не создан)")

analyze.py - 2008 байт
report.md - 763 байт
run.py - 1573 байт
sales.csv - 10808 байт

===== report.md =====

# Отчёт по продажам
## Выручка по регионам
| Регион | Выручка |
|---|---|
| Новосибирск | 1,572,150.00 |
| Москва | 1,443,500.00 |
| Казань | 1,381,580.00 |
| СПб | 986,930.00 |

## Выручка по месяцам
| Месяц | Выручка |
|---|---|
| 2026-04 | 982,810.00 |
| 2026-03 | 954,250.00 |
| 2026-06 | 910,590.00 |
| 2026-05 | 853,350.00 |
| 2026-02 | 848,080.00 |
| 2026-01 | 835,080.00 |

## Выводы
- Лидер по выручке — **Новосибирск** с 1,572,150.00. В топ-3 также входят: Москва, Казань.
- Самый прибыльный месяц — **2026-04** с 982,810.00. В топ-3 месяца: 2026-03, 2026-06.



## 6. Навыки (skills)

Кладём в отдельную песочницу два навыка — `sales-report` (толстый стандарт оформления + шаблон `template.md` рядом) и `data-quality-check`. В контексте у агента постоянно только их индекс (имя + описание); под задачу «подготовь отчёт» он сам выберет `sales-report`, прочитает его `SKILL.md` и `template.md` и оформит отчёт по стандарту. Так виден смысл скиллов: громоздкие процедуры не висят в промпте, а подгружаются по требованию.


In [7]:
import shutil
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import MemorySaver

# Отдельная чистая песочница для примера с навыками — без артефактов основного прогона.
skill_ws = Path("workspace_skill")
shutil.rmtree(skill_ws, ignore_errors=True)   # чистим песочницу навыков перед прогоном
skill_ws.mkdir(exist_ok=True)
shutil.copy(workspace / "sales.csv", skill_ws / "sales.csv")

skills_root = skill_ws / "skills"

# --- Навык 1: стандарт отчёта о продажах (со ссылкой на шаблон рядом) ---
sr = skills_root / "sales-report"
sr.mkdir(parents=True, exist_ok=True)
(sr / "SKILL.md").write_text("""---
name: sales-report
description: "Стандарт оформления отчёта о продажах компании. Используй при подготовке любого отчёта или сводки по продажам из табличных данных."
---

# Стандарт отчёта о продажах

Готовый отчёт сохрани в `report.md`. Структуру бери строго из `template.md` в этой же папке навыка: сначала прочитай шаблон, затем заполни разделы реальными числами.

## Обязательные разделы (в этом порядке)

1. **Ключевые цифры** — список: общая выручка, лучший регион, лучший месяц, средний чек.
2. **Выручка по регионам** — таблица, сортировка по убыванию выручки. Колонки: Регион, Выручка, Доля.
3. **Выручка по месяцам** — таблица в хронологическом порядке. Колонки: Месяц, Выручка, Прирост к предыдущему.
4. **Наблюдения** — 2-4 вывода строго из данных.

## Оформление чисел

- Суммы — в рублях, целые, разделитель разрядов пробелом: `1 572 150 руб.`.
- Доли и приросты — в процентах с одним знаком после запятой: `12,3 %`. Отрицательный прирост — со знаком минус.

## Правила

- Только факты из данных: никаких прогнозов и оценок «на глаз».
- Если в данных есть пропуски или явные выбросы — отметь одной строкой в «Наблюдениях», но отчёт всё равно построй.
""", encoding="utf-8")
(sr / "template.md").write_text("""# Отчёт о продажах

## Ключевые цифры
- Общая выручка: <сумма> руб.
- Лучший регион: <регион> (<сумма> руб.)
- Лучший месяц: <YYYY-MM> (<сумма> руб.)
- Средний чек: <сумма> руб.

## Выручка по регионам
| Регион | Выручка | Доля |
|---|---|---|
| ... | ... | ... |

## Выручка по месяцам
| Месяц | Выручка | Прирост к пред. |
|---|---|---|
| ... | ... | ... |

## Наблюдения
- ...
""", encoding="utf-8")

# --- Навык 2: проверка качества данных (чтобы агент выбирал нужный из индекса) ---
dq = skills_root / "data-quality-check"
dq.mkdir(parents=True, exist_ok=True)
(dq / "SKILL.md").write_text("""---
name: data-quality-check
description: "Проверка качества табличных данных перед анализом: дубликаты, пропуски, битые типы, выбросы. Используй, когда просят проверить или почистить данные."
---

# Проверка качества данных

Прогони по данным чек-лист и оформи короткий `data_quality.md`:

1. **Дубликаты** — полностью совпадающие строки, сколько.
2. **Пропуски** — по каждой колонке число пустых значений.
3. **Типы** — числовые колонки действительно числовые, даты парсятся.
4. **Выбросы** — значения за пределами разумного (например, отрицательные qty или price).

По каждому пункту — «OK» или что нашлось. В конце — вердикт: годны ли данные к анализу.
""", encoding="utf-8")

skill_backend = LocalShellBackend(root_dir=str(skill_ws.resolve()), virtual_mode=True)

agent = create_deep_agent(
    model=llm,
    backend=skill_backend,
    skills=["/skills"],  # виртуальный путь: корень бэкенда = skill_ws
    checkpointer=MemorySaver(),
    system_prompt=(
        "Ты — аналитик данных с доступом к файлам и shell. "
        "Работай итеративно: изучи данные, напиши код, запусти его, "
        "проверь результат. Используй относительные пути. "
        "Не выдумывай цифры — бери их только из вывода своих скриптов."
    ),
)

config = {"configurable": {"thread_id": "sales-report-skill"}}
result = agent.invoke({"messages": [{"role": "user", "content":
    "Подготовь отчёт по продажам из sales.csv по нашим правилам оформления."}]}, config=config)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Подготовь отчёт по продажам из sales.csv по нашим правилам оформления.
================================== Ai Message ==================================
Tool Calls:
  read_file (e452688b-ed9e-4fe2-99aa-94f2377b957e)
 Call ID: e452688b-ed9e-4fe2-99aa-94f2377b957e
  Args:
    file_path: /skills/sales-report/SKILL.md
================================= Tool Message =================================
Name: read_file

     1	---
     2	name: sales-report
     3	description: "Стандарт оформления отчёта о продажах компании. Используй при подготовке любого отчёта или сводки по продажам из табличных данных."
     4	---
     5	
     6	# Стандарт отчёта о продажах
     7	
     8	Готовый отчёт сохрани в `report.md`. Структуру бери строго из `template.md` в этой же папке навыка: сначала прочитай шаблон, затем заполни разделы реальными числами.
     9	
    10	## Обязательные разделы (в этом порядке)
    11	
    12	1. **Клю